In [19]:
!pip install -q requests beautifulsoup4 pytest

In [26]:
!pip install -q requests beautifulsoup4 pandas pytest

import os
import re
import csv
import time
import json
import hashlib
from datetime import datetime, timedelta, timezone
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import xml.etree.ElementTree as ET

import requests
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import files


# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://www.techi.com"
ROBOTS_URL = BASE_URL + "/robots.txt"

OUTPUT_CSV = "techi_articles.csv"
CACHE_DIR = "techi_cache"

os.makedirs(CACHE_DIR, exist_ok=True)

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


# ============================================================
# SESSION
# ============================================================

session = requests.Session()
session.headers.update(HEADERS)

last_request_time = 0


# ============================================================
# ROBOTS.TXT
# ============================================================

robot_parser = RobotFileParser()

ROBOTS_AVAILABLE = False


def load_robots():

    global ROBOTS_AVAILABLE

    try:

        print("Loading robots.txt...")

        response = session.get(
            ROBOTS_URL,
            timeout=15
        )

        response.raise_for_status()

        robot_parser.parse(
            response.text.splitlines()
        )

        ROBOTS_AVAILABLE = True

        print("robots.txt loaded.")

        return response.text

    except Exception as e:

        print(
            f"Could not load robots.txt: {e}"
        )

        ROBOTS_AVAILABLE = False

        return ""


robots_text = load_robots()


# ============================================================
# ROBOTS CHECK
# ============================================================

def allowed_by_robots(url):

    if not ROBOTS_AVAILABLE:
        return False

    try:
        return robot_parser.can_fetch(
            USER_AGENT,
            url
        )

    except Exception:
        return False


# ============================================================
# CACHE
# ============================================================

def cache_path(url):

    key = hashlib.sha256(
        url.encode("utf-8")
    ).hexdigest()

    return os.path.join(
        CACHE_DIR,
        key + ".cache"
    )


def load_cache(url):

    path = cache_path(url)

    if not os.path.exists(path):
        return None

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            return f.read()

    except Exception:
        return None


def save_cache(url, text):

    try:

        with open(
            cache_path(url),
            "w",
            encoding="utf-8"
        ) as f:

            f.write(text)

    except Exception as e:

        print(
            f"Cache error: {e}"
        )


# ============================================================
# SAFE REQUEST
# ============================================================

def safe_get(url, use_cache=True):

    global last_request_time

    # ----------------------------------------
    # CACHE FIRST
    # ----------------------------------------

    if use_cache:

        cached = load_cache(url)

        if cached is not None:

            print(
                f"  CACHE: {url}"
            )

            return cached

    # ----------------------------------------
    # ROBOTS
    # ----------------------------------------

    if not allowed_by_robots(url):

        print(
            f"  ROBOTS BLOCKED: {url}"
        )

        return None

    # ----------------------------------------
    # 1 REQUEST / SECOND
    # ----------------------------------------

    elapsed = (
        time.time()
        - last_request_time
    )

    if elapsed < 1:

        time.sleep(
            1 - elapsed
        )

    try:

        print(
            f"  GET: {url}"
        )

        response = session.get(
            url,
            timeout=15
        )

        last_request_time = time.time()

        if response.status_code == 200:

            text = response.text

            save_cache(
                url,
                text
            )

            return text

        if response.status_code == 404:

            print(
                f"  404: {url}"
            )

            return None

        print(
            f"  HTTP {response.status_code}: {url}"
        )

    except requests.Timeout:

        print(
            f"  TIMEOUT: {url}"
        )

    except requests.RequestException as e:

        print(
            f"  REQUEST ERROR: {e}"
        )

    except Exception as e:

        print(
            f"  ERROR: {e}"
        )

    return None


# ============================================================
# ARTICLE URL
# ============================================================

def is_article_url(url):

    try:

        parsed = urlparse(url)

        hostname = (
            parsed.netloc
            .lower()
            .split(":")[0]
        )

        if hostname not in [
            "techi.com",
            "www.techi.com"
        ]:
            return False

        path = parsed.path.strip("/")

        if not path:
            return False

        # Files are not articles
        if "." in path.split("/")[-1]:
            return False

        segments = [
            x for x in path.split("/")
            if x
        ]

        # TECHi article URLs are normally:
        # /some-article-slug/
        if len(segments) != 1:
            return False

        excluded = {
            "category",
            "categories",
            "tag",
            "tags",
            "author",
            "authors",
            "topic",
            "topics",
            "about",
            "contact",
            "privacy-policy",
            "terms",
            "cookies",
            "disclaimer",
            "feed",
            "search"
        }

        if segments[0].lower() in excluded:
            return False

        return True

    except Exception:
        return False


# ============================================================
# XML LOC EXTRACTION
# ============================================================

def extract_locs(xml_text):

    urls = []

    try:

        root = ET.fromstring(
            xml_text
        )

        for element in root.iter():

            tag = element.tag

            if "}" in tag:
                tag = tag.split(
                    "}",
                    1
                )[1]

            if tag.lower() == "loc":

                value = (
                    element.text or ""
                ).strip()

                if value:
                    urls.append(value)

    except Exception as e:

        print(
            f"  XML parse error: {e}"
        )

    return urls


# ============================================================
# DISCOVER SITEMAPS FROM ROBOTS
# ============================================================

def get_sitemaps_from_robots():

    sitemaps = []

    for line in robots_text.splitlines():

        line = line.strip()

        if line.lower().startswith(
            "sitemap:"
        ):

            value = line.split(
                ":",
                1
            )[1].strip()

            value = urljoin(
                BASE_URL,
                value
            )

            if value not in sitemaps:

                sitemaps.append(
                    value
                )

    return sitemaps


# ============================================================
# DISCOVER ARTICLES
# ============================================================

def discover_urls(limit=20):

    discovered = []

    sitemap_urls = (
        get_sitemaps_from_robots()
    )

    print(
        "\nSitemaps from robots.txt:"
    )

    for sm in sitemap_urls:
        print(
            " ",
            sm
        )

    if not sitemap_urls:

        print(
            "No sitemap found in robots.txt."
        )

        return []

    # --------------------------------------------------------
    # Process sitemap index / sitemap files
    # --------------------------------------------------------

    queue = list(sitemap_urls)
    processed_sitemaps = set()

    while queue and len(discovered) < limit:

        sitemap_url = queue.pop(0)

        if sitemap_url in processed_sitemaps:
            continue

        processed_sitemaps.add(
            sitemap_url
        )

        print(
            f"\nProcessing sitemap: {sitemap_url}"
        )

        xml = safe_get(
            sitemap_url
        )

        if not xml:
            continue

        locs = extract_locs(
            xml
        )

        for loc in locs:

            # ------------------------------------------------
            # Nested sitemap
            # ------------------------------------------------

            if (
                "sitemap" in loc.lower()
                and loc.lower().endswith(".xml")
            ):

                if loc not in processed_sitemaps:

                    queue.append(
                        loc
                    )

                continue

            # ------------------------------------------------
            # Article
            # ------------------------------------------------

            if is_article_url(loc):

                if loc not in discovered:

                    discovered.append(
                        loc
                    )

                    print(
                        f"  ARTICLE {len(discovered)}: {loc}"
                    )

                    if len(discovered) >= limit:
                        break

    return discovered[:limit]


# ============================================================
# DATE PARSER
# ============================================================

def parse_relative_or_absolute_date(
    date_str,
    reference_dt=None
):

    if not date_str:
        return ""

    if not isinstance(
        date_str,
        str
    ):
        return ""

    clean = re.sub(
        r"\s+",
        " ",
        date_str
    ).strip()

    if not clean:
        return ""

    ref = (
        reference_dt
        or datetime.now(timezone.utc)
    )

    # Updated 6 days ago
    # Published 2 hours ago
    # Posted 10 minutes ago

    clean = re.sub(
        r"^(updated|published|posted|modified|"
        r"last updated|last modified)"
        r"\s*(on|at)?\s*:?\s*",
        "",
        clean,
        flags=re.I
    ).strip()

    # --------------------------------------------------------
    # Relative
    # --------------------------------------------------------

    pattern = re.compile(
        r"(\d+)\s*"
        r"(seconds?|secs?|"
        r"minutes?|mins?|"
        r"hours?|hrs?|"
        r"days?|"
        r"weeks?|"
        r"months?|"
        r"years?)"
        r"\s+ago",
        re.I
    )

    match = pattern.search(
        clean
    )

    if match:

        qty = int(
            match.group(1)
        )

        unit = (
            match.group(2)
            .lower()
        )

        if unit.startswith("sec"):
            delta = timedelta(
                seconds=qty
            )

        elif unit.startswith("min"):
            delta = timedelta(
                minutes=qty
            )

        elif (
            unit.startswith("hour")
            or unit.startswith("hr")
        ):
            delta = timedelta(
                hours=qty
            )

        elif unit.startswith("day"):
            delta = timedelta(
                days=qty
            )

        elif unit.startswith("week"):
            delta = timedelta(
                weeks=qty
            )

        elif unit.startswith("month"):
            delta = timedelta(
                days=qty * 30
            )

        elif unit.startswith("year"):
            delta = timedelta(
                days=qty * 365
            )

        else:
            return ""

        return (
            ref - delta
        ).astimezone(
            timezone.utc
        ).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        )

    if re.search(
        r"\byesterday\b",
        clean,
        re.I
    ):

        return (
            ref - timedelta(days=1)
        ).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        )

    if re.search(
        r"\btoday\b",
        clean,
        re.I
    ):

        return ref.strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        )

    # --------------------------------------------------------
    # Ordinal dates
    # --------------------------------------------------------

    clean = re.sub(
        r"(\d{1,2})(st|nd|rd|th)\b",
        r"\1",
        clean,
        flags=re.I
    )

    # --------------------------------------------------------
    # ISO
    # --------------------------------------------------------

    try:

        iso_value = clean

        if iso_value.endswith("Z"):

            iso_value = (
                iso_value[:-1]
                + "+00:00"
            )

        dt = datetime.fromisoformat(
            iso_value
        )

        if dt.tzinfo is None:

            dt = dt.replace(
                tzinfo=timezone.utc
            )

        else:

            dt = dt.astimezone(
                timezone.utc
            )

        return dt.strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        )

    except ValueError:
        pass

    # --------------------------------------------------------
    # Normal dates
    # --------------------------------------------------------

    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%m/%d/%Y",
        "%d/%m/%Y",
        "%B %d, %Y",
        "%b %d, %Y",
        "%d %B %Y",
        "%d %b %Y",
        "%B %d %Y",
        "%b %d %Y",
        "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M:%S",
    ]

    for fmt in formats:

        try:

            dt = datetime.strptime(
                clean,
                fmt
            )

            dt = dt.replace(
                tzinfo=timezone.utc
            )

            return dt.strftime(
                "%Y-%m-%dT%H:%M:%SZ"
            )

        except ValueError:
            continue

    return ""


# ============================================================
# AUTHOR HANDLE
# ============================================================

def extract_author_handle(soup):

    # --------------------------------------------------------
    # Author URL
    # --------------------------------------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a.get(
            "href",
            ""
        )

        match = re.search(
            r"/author/([^/?#]+)/?",
            href,
            re.I
        )

        if match:

            return match.group(1).strip()

    # --------------------------------------------------------
    # JSON-LD
    # --------------------------------------------------------

    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):

        try:

            data = json.loads(
                script.string
                or script.get_text()
            )

            objects = []

            if isinstance(
                data,
                dict
            ):

                objects.append(
                    data
                )

                if isinstance(
                    data.get("@graph"),
                    list
                ):

                    objects.extend(
                        data["@graph"]
                    )

            elif isinstance(
                data,
                list
            ):

                objects.extend(
                    data
                )

            for item in objects:

                if not isinstance(
                    item,
                    dict
                ):
                    continue

                author = item.get(
                    "author"
                )

                if isinstance(
                    author,
                    dict
                ):

                    author_url = author.get(
                        "url",
                        ""
                    )

                    match = re.search(
                        r"/author/([^/?#]+)/?",
                        str(author_url),
                        re.I
                    )

                    if match:
                        return match.group(1)

                    name = author.get(
                        "name",
                        ""
                    )

                    if name:
                        return str(
                            name
                        ).strip()

                elif author:

                    if isinstance(
                        author,
                        list
                    ):

                        author = (
                            author[0]
                            if author
                            else ""
                        )

                    if isinstance(
                        author,
                        dict
                    ):

                        name = author.get(
                            "name",
                            ""
                        )

                        if name:
                            return str(
                                name
                            ).strip()

                    elif isinstance(
                        author,
                        str
                    ):

                        return author.strip()

        except Exception:
            continue

    # --------------------------------------------------------
    # Meta author
    # --------------------------------------------------------

    meta = soup.find(
        "meta",
        attrs={
            "name": re.compile(
                r"author",
                re.I
            )
        }
    )

    if meta:

        value = meta.get(
            "content",
            ""
        ).strip()

        if value:
            return value

    return ""


# ============================================================
# DATE EXTRACTION
# ============================================================

def extract_date_text(soup):

    candidates = []

    # JSON-LD
    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):

        try:

            data = json.loads(
                script.string
                or script.get_text()
            )

            objects = []

            if isinstance(
                data,
                dict
            ):

                objects.append(
                    data
                )

                if isinstance(
                    data.get("@graph"),
                    list
                ):

                    objects.extend(
                        data["@graph"]
                    )

            elif isinstance(
                data,
                list
            ):

                objects.extend(
                    data
                )

            for item in objects:

                if not isinstance(
                    item,
                    dict
                ):
                    continue

                for key in [
                    "datePublished",
                    "dateModified",
                    "dateCreated"
                ]:

                    value = item.get(
                        key
                    )

                    if value:
                        candidates.append(
                            str(value)
                        )

        except Exception:
            continue

    # <time>
    for tag in soup.find_all(
        "time"
    ):

        value = (
            tag.get("datetime")
            or tag.get("content")
            or tag.get_text(
                " ",
                strip=True
            )
        )

        if value:
            candidates.append(
                value
            )

    # meta
    for attrs in [
        {"property": "article:published_time"},
        {"property": "article:modified_time"},
        {"property": "og:published_time"},
        {"name": "date"},
        {"name": "publish_date"},
        {"name": "published_date"},
    ]:

        tag = soup.find(
            "meta",
            attrs=attrs
        )

        if tag:

            value = tag.get(
                "content",
                ""
            )

            if value:
                candidates.append(
                    value
                )

    # Classes
    for tag in soup.find_all(
        class_=re.compile(
            r"date|published|updated|posted|timestamp",
            re.I
        )
    ):

        value = tag.get_text(
            " ",
            strip=True
        )

        if value:
            candidates.append(
                value
            )

    # Return first parseable
    for candidate in candidates:

        if parse_relative_or_absolute_date(
            candidate
        ):

            return candidate

    return ""


# ============================================================
# ARTICLE EXTRACTION
# ============================================================

def extract_article(
    url,
    html
):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    path = urlparse(
        url
    ).path.strip("/")

    slug = (
        path.split("/")[-1]
        if path
        else ""
    )

    title = ""
    category = ""

    # --------------------------------------------------------
    # JSON-LD
    # --------------------------------------------------------

    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):

        try:

            data = json.loads(
                script.string
                or script.get_text()
            )

            objects = []

            if isinstance(
                data,
                dict
            ):

                objects.append(
                    data
                )

                if isinstance(
                    data.get("@graph"),
                    list
                ):

                    objects.extend(
                        data["@graph"]
                    )

            elif isinstance(
                data,
                list
            ):

                objects.extend(
                    data
                )

            for item in objects:

                if not isinstance(
                    item,
                    dict
                ):
                    continue

                types = item.get(
                    "@type",
                    []
                )

                if isinstance(
                    types,
                    str
                ):
                    types = [types]

                if not any(
                    x in [
                        "Article",
                        "NewsArticle",
                        "BlogPosting"
                    ]
                    for x in types
                ):
                    continue

                if not title:

                    title = (
                        item.get(
                            "headline"
                        )
                        or item.get(
                            "name"
                        )
                        or ""
                    )

                if not category:

                    section = item.get(
                        "articleSection"
                    )

                    if isinstance(
                        section,
                        list
                    ):

                        category = (
                            section[0]
                            if section
                            else ""
                        )

                    elif section:

                        category = str(
                            section
                        )

        except Exception:
            continue

    # --------------------------------------------------------
    # Title fallback
    # --------------------------------------------------------

    if not title:

        h1 = soup.find(
            "h1"
        )

        if h1:

            title = h1.get_text(
                " ",
                strip=True
            )

    if not title and soup.title:

        title = soup.title.get_text(
            " ",
            strip=True
        )

    # --------------------------------------------------------
    # Category fallback
    # --------------------------------------------------------

    if not category:

        meta = soup.find(
            "meta",
            property=re.compile(
                r"article:section",
                re.I
            )
        )

        if meta:

            category = meta.get(
                "content",
                ""
            )

    if not category:

        tag = soup.find(
            "a",
            href=re.compile(
                r"/(category|topic|section)/",
                re.I
            )
        )

        if tag:

            category = tag.get_text(
                " ",
                strip=True
            )

    # --------------------------------------------------------
    # Author
    # --------------------------------------------------------

    author_handle = (
        extract_author_handle(
            soup
        )
    )

    # --------------------------------------------------------
    # Date
    # --------------------------------------------------------

    date_text = extract_date_text(
        soup
    )

    date_iso = (
        parse_relative_or_absolute_date(
            date_text
        )
    )

    return {
        "url": url,
        "slug": slug,
        "title": title.strip(),
        "category": category.strip(),
        "author_handle": author_handle.strip(),
        "date_text": date_text.strip(),
        "date_iso": date_iso.strip(),
    }


# ============================================================
# MAIN
# ============================================================

urls = discover_urls(
    limit=20
)

print(
    "\n================================"
)

print(
    f"DISCOVERED ARTICLES: {len(urls)}"
)

print(
    "================================"
)

if not urls:

    print(
        "\nNo articles discovered."
    )

else:

    rows = []

    for i, url in enumerate(
        urls,
        1
    ):

        print(
            f"\n[{i}/{len(urls)}] {url}"
        )

        html = safe_get(
            url
        )

        if not html:
            continue

        try:

            data = extract_article(
                url,
                html
            )

            rows.append(
                data
            )

            print(
                "  Title:",
                data["title"]
            )

            print(
                "  Author:",
                data["author_handle"]
            )

            print(
                "  Date:",
                data["date_text"]
            )

            print(
                "  ISO:",
                data["date_iso"]
            )

        except Exception as e:

            print(
                f"  Extraction error: {e}"
            )

    columns = [
        "url",
        "slug",
        "title",
        "category",
        "author_handle",
        "date_text",
        "date_iso"
    ]

    with open(
        OUTPUT_CSV,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=columns
        )

        writer.writeheader()
        writer.writerows(rows)

    print(
        f"\nSaved {len(rows)} articles "
        f"to {OUTPUT_CSV}"
    )

    df = pd.read_csv(
        OUTPUT_CSV
    )

    display(df)

    files.download(
        OUTPUT_CSV
    )

Loading robots.txt...
robots.txt loaded.

Sitemaps from robots.txt:
  https://www.techi.com/sitemap_index.xml

Processing sitemap: https://www.techi.com/sitemap_index.xml
  GET: https://www.techi.com/sitemap_index.xml

Processing sitemap: https://www.techi.com/post-sitemap1.xml
  GET: https://www.techi.com/post-sitemap1.xml
  ARTICLE 1: https://www.techi.com/gpt-6-astra-openai-computer-use/
  ARTICLE 2: https://www.techi.com/deepmind-coding-research-california-london-cluster/
  ARTICLE 3: https://www.techi.com/amd-chip-selloff-input-costs-pricing-power/
  ARTICLE 4: https://www.techi.com/tesla-cybercab-hardware-gap-fsd-owners/
  ARTICLE 5: https://www.techi.com/meta-29-state-trial-advisory-jury-remedy-risk/
  ARTICLE 6: https://www.techi.com/nvidia-h200-china-shipments-below-licence-ceiling/
  ARTICLE 7: https://www.techi.com/samsung-foundry-price-hike-losing-share/
  ARTICLE 8: https://www.techi.com/ether-price-first-half-2026/
  ARTICLE 9: https://www.techi.com/anthropic-model-2-risk

,url,slug,title,category,author_handle,date_text,date_iso
0,https://www.techi.com/gpt-6-astra-openai-compu...,gpt-6-astra-openai-computer-use,GPT-6 Astra Reaches Human-Level Computer Use —...,"AI & Intelligence Models, agents, chips, labs,...",Saba Javed,2026-09-03T23:49:28.857Z,2026-09-03T23:49:28Z
1,https://www.techi.com/deepmind-coding-research...,deepmind-coding-research-california-london-clu...,Google DeepMind's shift to California tests th...,AI & Intelligence,Zoha Imdad Ali,2026-08-19T23:24:43.152Z,2026-08-19T23:24:43Z
2,https://www.techi.com/amd-chip-selloff-input-c...,amd-chip-selloff-input-costs-pricing-power,"The chip selloff is picking sides, and AMD is ...","AI & Intelligence Models, agents, chips, labs,...",Umair Aslam,2026-08-19T20:41:39.259Z,2026-08-19T20:41:39Z
3,https://www.techi.com/tesla-cybercab-hardware-...,tesla-cybercab-hardware-gap-fsd-owners,Tesla's robotaxi finally works. Just not on th...,"AI & Intelligence Models, agents, chips, labs,...",Omer Sheikh,2026-08-19T20:22:24.233Z,2026-08-19T20:22:24Z
4,https://www.techi.com/meta-29-state-trial-advi...,meta-29-state-trial-advisory-jury-remedy-risk,The Meta trial's real risk isn't the $1.4 tril...,"AI & Intelligence Models, agents, chips, labs,...",Saba Javed,2026-08-19T20:04:04.214Z,2026-08-19T20:04:04Z
5,https://www.techi.com/nvidia-h200-china-shipme...,nvidia-h200-china-shipments-below-licence-ceiling,"Nvidia's H200s are flowing to China again, at ...","AI & Intelligence Models, agents, chips, labs,...",Dr Layloma Rashid,2026-08-19T19:07:41.212Z,2026-08-19T19:07:41Z
6,https://www.techi.com/samsung-foundry-price-hi...,samsung-foundry-price-hike-losing-share,Samsung lost a third of its foundry share. It ...,"AI & Intelligence Models, agents, chips, labs,...",Fatimah Misbah Hussain,2026-08-19T18:14:26.171Z,2026-08-19T18:14:26Z
7,https://www.techi.com/ether-price-first-half-2...,ether-price-first-half-2026,Reading the Ether Price After a Brutal First H...,Markets & Equities,Qaiser Sultan,2026-08-19T17:03:28.994Z,2026-08-19T17:03:28Z
8,https://www.techi.com/anthropic-model-2-risk-r...,anthropic-model-2-risk-report-misalignment-est...,Anthropic’s Model 2 Is Stronger. That Isn’t Wh...,"AI & Intelligence Models, agents, chips, labs,...",Qaiser Sultan,2026-08-14T22:43:01.312Z,2026-08-14T22:43:01Z
9,https://www.techi.com/intel-15-billion-stock-o...,intel-15-billion-stock-offering,Intel Just Asked Investors for $15 Billion. Th...,"AI & Intelligence Models, agents, chips, labs,...",Omer Sheikh,2026-08-10T18:17:58.730Z,2026-08-10T18:17:58Z


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>